Install required packages

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy

Import the libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving thecourseforum_all_reviews (1).csv to thecourseforum_all_reviews (1) (1).csv


Load the dataset

In [ ]:
reviews = pd.read_csv("/content/thecourseforum_all_reviews (1).csv")

print("Dataset shape:", reviews.shape)
print("\nColumns:")
print(reviews.columns.tolist())

Dataset shape: (4699, 18)

Columns:
['review_id', 'department', 'course_code', 'course_title', 'instructor_name', 'semester_taken', 'review_date', 'overall_rating', 'difficulty_rating', 'instructor_rating', 'enjoyability_rating', 'recommend_rating', 'hours_per_week', 'upvotes', 'review_text', 'review_url', 'source_url', 'scraped_at']


Check the two columns

In [ ]:
print("Missing review_text:", reviews["review_text"].isna().sum())
print("Missing difficulty_rating:", reviews["difficulty_rating"].isna().sum())

print("\nDifficulty rating distribution:")
print(reviews["difficulty_rating"].value_counts().sort_index())

print("\nFirst 3 reviews:")
print(reviews["review_text"].head(3))

Missing review_text: 0
Missing difficulty_rating: 0

Difficulty rating distribution:
difficulty_rating
1.0     484
2.0    1113
3.0    1380
4.0    1210
5.0     512
Name: count, dtype: int64

First 3 reviews:
0    Fall 2021 5.0 AVERAGE This is the first langua...
1    Fall 2015 4.7 AVERAGE I highly Dkt. Anne. She ...
2    Fall 2023 3.3 AVERAGE If you asked me mid-seme...
Name: review_text, dtype: object


Inspect the raw review structure

In [ ]:
for i in range(5):
    print(f"\n--- REVIEW {i} ---")
    print(reviews.loc[i, "review_text"])


--- REVIEW 0 ---
Fall 2021 5.0 AVERAGE This is the first language class that I have enjoyed, ever. I finished up the language requirement for the College in my second year with another language and decided to take Swahili not as a requirement, but purely out of interest. This was one of the best decisions I have made as a student. Professor Rotich is amazing at what she does, and she finds a way to make learning a new language and the culture of its large speaker base exciting and intuitive. Never once did I feel like the class was going too fast or that I had missed something integral to later learning and was behind. We learned how to speak, write, and read in Swahili at a beginner level, and started to learn some of the various noun classes Swahili has, which I assume will be a more central topic in higher levels. If you are looking for a fun class to add to your course load, consider Swahili with Prof. Rotich! 1 Jan. 16, 2022

--- REVIEW 1 ---
Fall 2015 4.7 AVERAGE I highly Dkt. A

Check the beginning and ending patterns across the dataset

In [ ]:
print("FIRST 10 REVIEW TEXTS:")
print("=" * 80)

for i in range(10):
    text = reviews.loc[i, "review_text"]
    print(f"\nReview {i}:")
    print(text[:150])
    print("...")
    print(text[-100:])

FIRST 10 REVIEW TEXTS:

Review 0:
Fall 2021 5.0 AVERAGE This is the first language class that I have enjoyed, ever. I finished up the language requirement for the College in my second 
...
king for a fun class to add to your course load, consider Swahili with Prof. Rotich! 1 Jan. 16, 2022

Review 1:
Fall 2015 4.7 AVERAGE I highly Dkt. Anne. She explains material thoroughly, and also intertwines the information with its cultural context. In additio
...
ances. She truly cares that her students are learning in a well rounded environment. 2 April 5, 2016

Review 2:
Fall 2023 3.3 AVERAGE If you asked me mid-semester if I recommended the class, my answer would be no. However, post-class I definitely I will admit th
...
 of work a lot, which is great if you are also interested in her research or health. 1 July 25, 2024

Review 3:
Fall 2018 4.0 AVERAGE This course does a great job explaining the systemic targeting of minorities by the government. It shifts students away from the
...
 get ahead

Clean review text

In [ ]:
import re

def clean_review(text):
    # remove the webpage heading containing the term, year, and average rating.
    # January is included so January-term reviews are cleaned too.
    text = re.sub(
        r'^(Fall|Spring|Summer|Winter|January)\s+\d{4}\s+\d+(?:\.\d+)?\s+AVERAGE\s*',
        '',
        text,
        flags=re.IGNORECASE
    )

    # remove the trailing vote count and posting date collected from the webpage.
    # keep the student's actual writing between the heading and footer.
    text = re.sub(
        r'\s+-?\d+\s+'
        r'(?:Jan\.?|Feb\.?|Mar\.?|Apr\.?|May|Jun\.?|Jul\.?|Aug\.?|'
        r'Sept?\.?|Oct\.?|Nov\.?|Dec\.?|January|February|March|April|'
        r'June|July|August|September|October|November|December)'
        r'\s+\d{1,2},\s+\d{4}\s*$',
        '',
        text,
        flags=re.IGNORECASE
    )

    # remove extra spaces from the beginning and end of the cleaned review.
    return text.strip()

# create a cleaned column while preserving the original review_text column.
reviews["cleaned_review"] = reviews["review_text"].apply(clean_review)

Check that the cleaning worked

In [ ]:
for i in range(5):
    print(f"\n--- CLEANED REVIEW {i} ---")
    print(reviews.loc[i, "cleaned_review"])


--- CLEANED REVIEW 0 ---
This is the first language class that I have enjoyed, ever. I finished up the language requirement for the College in my second year with another language and decided to take Swahili not as a requirement, but purely out of interest. This was one of the best decisions I have made as a student. Professor Rotich is amazing at what she does, and she finds a way to make learning a new language and the culture of its large speaker base exciting and intuitive. Never once did I feel like the class was going too fast or that I had missed something integral to later learning and was behind. We learned how to speak, write, and read in Swahili at a beginner level, and started to learn some of the various noun classes Swahili has, which I assume will be a more central topic in higher levels. If you are looking for a fun class to add to your course load, consider Swahili with Prof. Rotich!

--- CLEANED REVIEW 1 ---
I highly Dkt. Anne. She explains material thoroughly, and a

Create the features and target

In [ ]:
# Text feature
X = reviews["cleaned_review"]

# Target variable
y = reviews["difficulty_rating"]

print("Number of reviews:", len(X))
print("Number of labels:", len(y))

print("\nTarget distribution:")
print(y.value_counts().sort_index())

Number of reviews: 4699
Number of labels: 4699

Target distribution:
difficulty_rating
1.0     484
2.0    1113
3.0    1380
4.0    1210
5.0     512
Name: count, dtype: int64


Create the 70/15/15 split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("Training set size:", len(X_train))
print("Temporary set size:", len(X_temp))

print("\nTraining rating distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTemporary rating distribution:")
print(y_temp.value_counts(normalize=True).sort_index())

Training set size: 3289
Temporary set size: 1410

Training rating distribution:
difficulty_rating
1.0    0.103071
2.0    0.236850
3.0    0.293706
4.0    0.257525
5.0    0.108848
Name: proportion, dtype: float64

Temporary rating distribution:
difficulty_rating
1.0    0.102837
2.0    0.236879
3.0    0.293617
4.0    0.257447
5.0    0.109220
Name: proportion, dtype: float64


Create validation and test sets

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training set size:", len(X_train))
print("Validation set size:", len(X_val))
print("Test set size:", len(X_test))

print("\nValidation rating distribution:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nTest rating distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Training set size: 3289
Validation set size: 705
Test set size: 705

Validation rating distribution:
difficulty_rating
1.0    0.102128
2.0    0.236879
3.0    0.293617
4.0    0.258156
5.0    0.109220
Name: proportion, dtype: float64

Test rating distribution:
difficulty_rating
1.0    0.103546
2.0    0.236879
3.0    0.293617
4.0    0.256738
5.0    0.109220
Name: proportion, dtype: float64


Load MiniLM

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.


Generate training embeddings

In [ ]:
X_train_embeddings = model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

print("Training embeddings shape:", X_train_embeddings.shape)

Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Training embeddings shape: (3289, 384)


Generate validation embeddings

In [ ]:
X_val_embeddings = model.encode(
    X_val.tolist(),
    show_progress_bar=True
)

print("Validation embeddings shape:", X_val_embeddings.shape)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Validation embeddings shape: (705, 384)


Generate test embeddings

In [ ]:
X_test_embeddings = model.encode(
    X_test.tolist(),
    show_progress_bar=True
)

print("Test embeddings shape:", X_test_embeddings.shape)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Test embeddings shape: (705, 384)


Final feature-engineering verification

In [ ]:
print("FINAL DATA CHECK")
print("-" * 40)

print("Training:")
print("  Embeddings:", X_train_embeddings.shape)
print("  Labels:", y_train.shape)

print("\nValidation:")
print("  Embeddings:", X_val_embeddings.shape)
print("  Labels:", y_val.shape)

print("\nTest:")
print("  Embeddings:", X_test_embeddings.shape)
print("  Labels:", y_test.shape)

FINAL DATA CHECK
----------------------------------------
Training:
  Embeddings: (3289, 384)
  Labels: (3289,)

Validation:
  Embeddings: (705, 384)
  Labels: (705,)

Test:
  Embeddings: (705, 384)
  Labels: (705,)


Export and download embeddings

In [ ]:
import numpy as np
from google.colab import files

np.savez_compressed(
    "embeddings_split.npz",
    X_train=X_train_embeddings,
    X_val=X_val_embeddings,
    X_test=X_test_embeddings,
    y_train=y_train.to_numpy(),
    y_val=y_val.to_numpy(),
    y_test=y_test.to_numpy()
)

files.download("embeddings_split.npz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>